In [ ]:
import sqlite3
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
#setting nltk libraries
nltk.download('vader_lexicon')
analyzer = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [ ]:
#uploading the database file
from google.colab import files
files.upload()

In [ ]:
conn = sqlite3.connect('data.sqlite')

In [ ]:
#testing to see if the tables are running
pd.read_sql('select * from "dbo.customer_reviews" LIMIT 5;', conn)

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,1,77,18,2023-12-23,3,"Average experience, nothing special."
1,2,80,19,2024-12-25,5,The quality is top-notch.
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper."
4,5,64,2,2023-07-16,3,"Average experience, nothing special."


In [ ]:
customer_reviews_df = pd.read_sql('SELECT ReviewID, CustomerID, ProductID, ReviewDate, Rating, ReviewText FROM "dbo.customer_reviews"', conn)


In [ ]:
print(customer_reviews_df)

     ReviewID CustomerID ProductID  ReviewDate Rating  \
0           1         77        18  2023-12-23      3   
1           2         80        19  2024-12-25      5   
2           3         50        13  2025-01-26      4   
3           4         78        15  2025-04-21      3   
4           5         64         2  2023-07-16      3   
...       ...        ...       ...         ...    ...   
1358     1359         28         4  2023-05-25      3   
1359     1360         58        12  2023-11-13      2   
1360     1361         96        15  2023-03-07      5   
1361     1362         99         2  2025-12-03      1   
1362     1363         16         4  2024-07-16      2   

                                      ReviewText  
0        Average  experience,  nothing  special.  
1                 The  quality  is    top-notch.  
2        Five  stars  for  the  quick  delivery.  
3       Good  quality,  but  could  be  cheaper.  
4        Average  experience,  nothing  special.  
...      

In [ ]:
#defining a function to calculate sentiment scores using vader
def calculate_sentiment(review):
  #getting sentiment scores for review text
  sentiment = analyzer.polarity_scores(review)
  #return compound score, which is normalised score between -1 and 1 (most negative to most postive respectfully)
  return sentiment['compound']

In [ ]:
#Define a function to categorise sentiment using both sentiment score and review rating
def categorize_sentiment(score, rating):
    #use both the text sentiment score and the numerical rating to determine sentiment category
    if score > 0.05:  #Positive Sentiment Score
      if rating >= 4:
          return 'Positive'   #High Rating and Positive Sentiment
      elif rating == 3:
          return 'Mixed Positive'   #Neutral Rating but Positive Sentiment
      else:
          return 'Mixed Negative'   #Low Rating bur Positive Sentiment
    elif score < -0.05:   #Negative Sentiment Score
        if rating <= 2:
            return 'Negative'   #Low Rating and Negative Sentiment
        elif rating == 3:
            return 'Mixed Negative'   #Neutral Rating but Negative Sentiment
        else:
          return 'Mixed Positive'   #High Rating but Negative Sentiment
    else:   #Neutral Sentiment Score
        if rating >= 4:
            return 'Positive'   #High rating but Neutral Sentiment
        elif rating <= 2:
            return 'Negative'   # Low rating with neutral sentiment
        else:
            return 'Neutral'    # Neutral rating and neutral sentiment



In [ ]:
#Defining a function to bucket sentiment scores into text ranges
def sentiment_bucket(score):
    if score >= 0.5:
        return '0.5 to 1.0'   #Strongly Positive Sentiment
    elif 0.0 <= score < 0.5:
        return '0.0 to 0.49'    #Mildly Positive Sentiment
    elif -0.5 <= score < 0.0:
        return '-0.49 to 0.0'   #Mildly Negative Sentiment
    else:
        return '-1.0 to -0.5'   #Strongly Negative Sentiment

In [ ]:
#Applying sentiment analysis to calculate sentiment scores for each review
customer_reviews_df['sentiment_score'] =  customer_reviews_df['ReviewText'].apply(calculate_sentiment)

#Applying sentiment bucket using both texting and rating
customer_reviews_df['sentiment_category'] = customer_reviews_df['sentiment_score'].apply(sentiment_bucket)

#display first few rows of the dataframe with sentiment scores, categories, and buckets
print(customer_reviews_df.head())

#saving dataframe with sentiment scores, categories, and buckets to a new CSV file
customer_reviews_df.to_csv('customer_reviews_with_sentiment.csv', index = False)

  ReviewID CustomerID ProductID  ReviewDate Rating  \
0        1         77        18  2023-12-23      3   
1        2         80        19  2024-12-25      5   
2        3         50        13  2025-01-26      4   
3        4         78        15  2025-04-21      3   
4        5         64         2  2023-07-16      3   

                                 ReviewText  sentiment_score  \
0   Average  experience,  nothing  special.          -0.3089   
1            The  quality  is    top-notch.           0.0000   
2   Five  stars  for  the  quick  delivery.           0.0000   
3  Good  quality,  but  could  be  cheaper.           0.2382   
4   Average  experience,  nothing  special.          -0.3089   

  sentiment_category  
0       -0.49 to 0.0  
1        0.0 to 0.49  
2        0.0 to 0.49  
3        0.0 to 0.49  
4       -0.49 to 0.0  
